# Nano-GraphRAG Integration

This notebook demonstrates how to use nano-graphrag as the core GraphRAG engine for Socratic AI Tutor.

## About nano-graphrag

[nano-graphrag](https://github.com/gusye1234/nano-graphrag) is a lightweight GraphRAG implementation that provides:
- Document indexing with automatic knowledge graph construction
- Local and global query modes
- Support for Ollama, OpenAI, and other LLM backends
- NetworkX for graph storage

## Hybrid Approach

We use a **hybrid approach**:
1. **nano-graphrag** for document retrieval and answering
2. **Custom NetworkX graph** for concept relationships, examples, and scaffolding

This gives us:
- ✅ Powerful retrieval from nano-graphrag
- ✅ Custom concept modeling for educational scaffolding
- ✅ Best of both worlds!

In [1]:
# Setup
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / "src"))

print(f"Project root: {project_root}")

Project root: <path> Assignments\CS 370\eng-ai-agents\project


## Option 1: Pure nano-graphrag

Use nano-graphrag directly for simple document Q&A.

In [2]:
from socratic.graph import NanoGraphRAGWrapper

# Initialize nano-graphrag
nano_rag = NanoGraphRAGWrapper(working_dir=project_root / "data" / "nano_graphrag")

print("✅ nano-graphrag initialized")

INFO:nano-graphrag:Load KV full_docs with 5 data
INFO:nano-graphrag:Load KV text_chunks with 7 data
INFO:nano-graphrag:Load KV llm_response_cache with 0 data
INFO:nano-graphrag:Load KV community_reports with 3 data
INFO:nano-graphrag:Loaded graph from ./data/nano_graphrag/graph_chunk_entity_relation.graphml with 142 nodes, 62 edges
INFO:nano-vectordb:Load (114, 768) data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': 'C:\\Users\\rjwil\\CodingProjects\\School Assignments\\CS 370\\eng-ai-agents\\project\\data\\nano_graphrag\\vdb_entities.json'} 114 data


✅ nano-graphrag initialized


In [3]:
# Load resources from previous ingestion
import json
from socratic.models import Resource

resources_file = project_root / "data" / "processed" / "ingested_resources.json"

if resources_file.exists():
    with open(resources_file, 'r') as f:
        resources_data = json.load(f)
    
    # Convert to Resource objects
    resources = [Resource(**r) for r in resources_data[:5]]  # Start with 5 for testing
    print(f"Loaded {len(resources)} resources")
else:
    print("⚠️ No resources found. Run 01_data_ingestion.ipynb first.")
    resources = []

Loaded 5 resources


In [4]:
# Insert resources into nano-graphrag
if resources:
    print("Inserting resources into nano-graphrag...")
    print("This may take a few minutes.\n")
    
    nano_rag.insert_resources(resources)
    
    print("\n✅ Resources inserted into nano-graphrag")

INFO:socratic.graph.nano_graphrag_wrapper:Inserting 5 resources into nano-graphrag...
INFO:nano-graphrag:Writing graph with 142 nodes, 62 edges
INFO:socratic.graph.nano_graphrag_wrapper:Inserted: Introduction to Artificial Intelligence – Engineering AI Agents
INFO:nano-graphrag:Writing graph with 142 nodes, 62 edges
INFO:socratic.graph.nano_graphrag_wrapper:Inserted: Back2classroom – Engineering AI Agents
INFO:nano-graphrag:Writing graph with 142 nodes, 62 edges
INFO:socratic.graph.nano_graphrag_wrapper:Inserted: Foundations – Engineering AI Agents
INFO:nano-graphrag:Writing graph with 142 nodes, 62 edges
INFO:socratic.graph.nano_graphrag_wrapper:Inserted: Training Deep Neural Networks – Engineering AI Agents
INFO:nano-graphrag:Writing graph with 142 nodes, 62 edges
INFO:socratic.graph.nano_graphrag_wrapper:Inserted: 2D Perception – Engineering AI Agents


Inserting resources into nano-graphrag...
This may take a few minutes.


✅ Resources inserted into nano-graphrag


In [5]:
# Query using nano-graphrag
question = "What is machine learning?"

print(f"Query: {question}\n")

# Try local mode (focused on nearby entities)
result_local = nano_rag.query(question, mode="local")
print("Local mode answer:")
print(result_local["answer"])
print("\n" + "="*80 + "\n")

# Try global mode (considers broader communities)
result_global = nano_rag.query(question, mode="global")
print("Global mode answer:")
print(result_global["answer"])

INFO:socratic.graph.nano_graphrag_wrapper:Querying nano-graphrag: What is machine learning?


Query: What is machine learning?



INFO:httpx:HTTP Request: POST http://localhost:11434/v1/embeddings "HTTP/1.1 200 OK"
INFO:nano-graphrag:Using 20 entites, 0 communities, 9 relations, 6 text units
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:socratic.graph.nano_graphrag_wrapper:Querying nano-graphrag: What is machine learning?
INFO:nano-graphrag:Revtrieved 3 communities
INFO:nano-graphrag:Grouping to 1 groups for global search


Local mode answer:
Machine learning is a subset of artificial intelligence (AI) that focuses on building systems and algorithms that enable computers to learn from data, identify patterns, and make predictions or decisions without being explicitly programmed. Here are key points about machine learning:

1. Definition: Machine learning involves training models using large amounts of data to make predictions or inferences.

2. Types:
   - Supervised learning ( Uses labeled training data)
   - Unsupervised learning (Uses unlabeled data, focuses on discovering patterns)
   - Semi-supervised learning (Combination of both)
   - Reinforcement learning (Involves rewards and punishments)

3. Key components:
   - Data collection
   - Feature selection
   - Model training
   - Evaluation
   - Deployment

4. Common applications:
   - Image and speech recognition
   - Natural language processing
   - Fraud detection
   - Recommender systems
   - Predictive analytics
   - Autonomous vehicles

5. Tec

INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"


Global mode answer:
Sorry, I'm not able to provide an answer to that question.


## Option 2: Hybrid Approach (Recommended)

Use both nano-graphrag AND custom concept graph for best results.

In [6]:
from socratic.graph import HybridKnowledgeGraphBuilder

# Initialize hybrid builder
hybrid_builder = HybridKnowledgeGraphBuilder(
    working_dir=project_root / "data" / "hybrid_graph"
)

print("✅ Hybrid builder initialized")

INFO:nano-graphrag:Load KV full_docs with 5 data
INFO:nano-graphrag:Load KV text_chunks with 7 data
INFO:nano-graphrag:Load KV llm_response_cache with 0 data
INFO:nano-graphrag:Load KV community_reports with 3 data
INFO:nano-graphrag:Loaded graph from ./data/hybrid_graph/graph_chunk_entity_relation.graphml with 113 nodes, 41 edges
INFO:nano-vectordb:Load (96, 768) data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': 'C:\\Users\\rjwil\\CodingProjects\\School Assignments\\CS 370\\eng-ai-agents\\project\\data\\hybrid_graph\\vdb_entities.json'} 96 data


✅ Hybrid builder initialized


In [7]:
# Build hybrid graph
if resources:
    print("Building hybrid knowledge graph...")
    print("This creates both nano-graphrag index AND custom concept graph.\n")
    
    kg = hybrid_builder.build_from_resources(resources)
    
    print(f"\n✅ Hybrid graph built:")
    print(f"   - nano-graphrag: Document indexing complete")
    print(f"   - Custom graph: {len(kg.concepts)} concepts, {len(kg.examples)} examples, {len(kg.edges)} edges")

INFO:socratic.graph.hybrid_builder:Building hybrid knowledge graph from 5 resources
INFO:socratic.graph.hybrid_builder:Step 1: Inserting into nano-graphrag...
INFO:socratic.graph.nano_graphrag_wrapper:Inserting 5 resources into nano-graphrag...
INFO:nano-graphrag:Writing graph with 113 nodes, 41 edges
INFO:socratic.graph.nano_graphrag_wrapper:Inserted: Introduction to Artificial Intelligence – Engineering AI Agents
INFO:nano-graphrag:Writing graph with 113 nodes, 41 edges
INFO:socratic.graph.nano_graphrag_wrapper:Inserted: Back2classroom – Engineering AI Agents
INFO:nano-graphrag:Writing graph with 113 nodes, 41 edges
INFO:socratic.graph.nano_graphrag_wrapper:Inserted: Foundations – Engineering AI Agents
INFO:nano-graphrag:Writing graph with 113 nodes, 41 edges
INFO:socratic.graph.nano_graphrag_wrapper:Inserted: Training Deep Neural Networks – Engineering AI Agents
INFO:nano-graphrag:Writing graph with 113 nodes, 41 edges
INFO:socratic.graph.nano_graphrag_wrapper:Inserted: 2D Perceptio

Building hybrid knowledge graph...
This creates both nano-graphrag index AND custom concept graph.



INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:socratic.graph.builder:Extracted 4 concepts from web_5279553432030179324
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:socratic.graph.builder:Extracted 0 examples from web_5279553432030179324
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:socratic.graph.builder:Identified 5 relationships
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:socratic.graph.builder:Extracted 4 concepts from web_4432039879397545034
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:socratic.graph.builder:Extracted 1 examples from web_4432039879397545034
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:socratic.graph.builder:Identified 3 relationships
INFO:httpx:HTTP Request: POST ht


✅ Hybrid graph built:
   - nano-graphrag: Document indexing complete
   - Custom graph: 20 concepts, 1 examples, 38 edges


In [8]:
# Get statistics
stats = hybrid_builder.get_stats()
print("Hybrid Graph Statistics:")
print(json.dumps(stats, indent=2))

Hybrid Graph Statistics:
{
  "nano_graphrag": {
    "working_dir": "C:\\Users\\rjwil\\CodingProjects\\School Assignments\\CS 370\\eng-ai-agents\\project\\data\\hybrid_graph",
    "has_graph": true,
    "has_embeddings": true
  },
  "custom_graph": {
    "concepts": 20,
    "resources": 5,
    "examples": 1,
    "edges": 38
  }
}


## Querying with Hybrid Engine

In [9]:
from socratic.query import HybridQueryEngine

# Create hybrid query engine
engine = HybridQueryEngine(hybrid_builder)

print("✅ Hybrid query engine ready")

INFO:socratic.graph.storage:NetworkX graph built: 26 nodes, 38 edges
INFO:socratic.graph.storage:NetworkX graph built: 26 nodes, 38 edges


✅ Hybrid query engine ready


In [10]:
# Query using nano-graphrag retrieval
question = "Explain neural networks"

result = engine.answer_query(
    question,
    use_nano_graphrag=True,
    mode="local"
)

print(f"Query: {question}\n")
print("Answer:")
print(result["answer"])
print(f"\nConcepts: {result['concepts_used']}")
print(f"Source: {result['source']}")

INFO:socratic.query.hybrid_engine:Processing hybrid query: Explain neural networks
INFO:socratic.graph.nano_graphrag_wrapper:Querying nano-graphrag: Explain neural networks
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/embeddings "HTTP/1.1 200 OK"
INFO:nano-graphrag:Using 20 entites, 2 communities, 5 relations, 4 text units
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:socratic.query.engine:Identified 3 relevant concepts for query
INFO:socratic.query.engine:Retrieved context: 3 concepts, 0 prerequisites, 3 resources, 0 examples


Query: Explain neural networks

Answer:
Neural networks are a type of machine learning model inspired by the structure and function of biological nervous systems, particularly the brain. They are used for a wide variety of tasks including image and speech recognition, natural language processing, recommendation systems, and more.

### Key Components

1. **Nodes (or Neurons):**
   - Each neuron takes input from previous nodes or external inputs.
   - Performs a computation using its weights and activation function to produce an output.
   - The outputs often serve as inputs for subsequent layers in the network.

2. **Layers:**
   - Layers are organized into three main types:
     - **Input Layer:** Consists of neurons representing the input features. It is the first layer after data enters the network.
     - **Hidden Layers:** These are intermediate layers between the input and output layers where most computations occur. They can vary in number, which determines the depth of the neura

In [11]:
# Query using custom scaffolded approach
result_scaffolded = engine.answer_with_scaffolding(
    "How do attention mechanisms work?"
)

print("Scaffolded Answer:")
print(result_scaffolded["answer"])
print(f"\nConcepts: {result_scaffolded['concepts_used']}")
print(f"\nCitations:")
for i, citation in enumerate(result_scaffolded['citations'], 1):
    print(f"  {i}. {citation['title']}")

INFO:socratic.query.hybrid_engine:Generating scaffolded answer for: How do attention mechanisms work?
INFO:socratic.query.engine:Processing query: How do attention mechanisms work?
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:socratic.query.engine:Identified 2 relevant concepts for query
INFO:socratic.query.engine:Retrieved context: 2 concepts, 1 prerequisites, 2 resources, 0 examples
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"


Scaffolded Answer:
{
  "answer": "Attention mechanisms are an essential part of recent advancements in machine learning, particularly in the field of Natural Language Processing (NLP). They allow models to focus on specific parts of input data when generating output. Let's start with a brief overview and then delve into how they work in detail.

### Prerequisite: Machine Learning
Machine Learning is a branch of Artificial Intelligence that focuses on developing algorithms which can learn from and make predictions on data without being explicitly programmed [Resource: Introduction to Artificial Intelligence – Engineering AI Agents, https://pantelis.github.io/courses/ai/in-person.html].

### Prerequisite: Natural Language Processing (NLP)
Natural Language Processing is a subfield of AI that deals with the interaction between computers and human languages. NLP involves tasks such as language translation, text summarization, and sentiment analysis [Resource: Introduction to Artificial Inte

## Compare Approaches

In [12]:
# Compare different query strategies
comparison = engine.compare_approaches(
    "What is the transformer architecture?"
)

print("Query:", comparison["query"])
print("\n" + "="*80)
print("\n1. nano-graphrag (local mode):")
print(comparison["nano_graphrag_local"]["answer"][:500] + "...")

print("\n" + "="*80)
print("\n2. nano-graphrag (global mode):")
print(comparison["nano_graphrag_global"]["answer"][:500] + "...")

print("\n" + "="*80)
print("\n3. Custom scaffolded:")
print(comparison["custom_scaffolded"]["answer"][:500] + "...")

INFO:socratic.query.hybrid_engine:Comparing query approaches for: What is the transformer architecture?
INFO:socratic.graph.nano_graphrag_wrapper:Querying nano-graphrag: What is the transformer architecture?
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/embeddings "HTTP/1.1 200 OK"
INFO:nano-graphrag:Using 20 entites, 3 communities, 9 relations, 4 text units
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:socratic.query.engine:Identified 2 relevant concepts for query
INFO:socratic.query.engine:Retrieved context: 2 concepts, 1 prerequisites, 2 resources, 0 examples
INFO:socratic.graph.nano_graphrag_wrapper:Querying nano-graphrag: What is the transformer architecture?
INFO:nano-graphrag:Revtrieved 3 communities
INFO:nano-graphrag:Grouping to 1 groups for global search
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "

Query: What is the transformer architecture?


1. nano-graphrag (local mode):
The Transformer architecture is a revolutionary model in the field of natural language processing (NLP) that has enabled significant advancements in various tasks, including translation, text summarization, and question-answering. It was introduced in the 2017 paper "Attention Is All You Need" by Vaswani et al.

### Key Components

1. **Self-Attention Mechanism**: 
   - Traditional recurrent neural networks (RNNs) like LSTM or GRU rely on sequential processing, which limits their parallelization...


2. nano-graphrag (global mode):
Sorry, I'm not able to provide an answer to that question....


3. Custom scaffolded:
{
  "answer": "The transformer architecture is a key component in modern Natural Language Processing (NLP) that has revolutionized how computers handle text data. To understand it better, let's start by reviewing some foundational concepts.

First, we need to know about Machine Learning [Resource:

## Evaluation Queries

Test with the three required evaluation queries.

In [ ]:
evaluation_queries = [
    "How do attention mechanisms work in transformers? Can you show me a Python implementation?",
    "What is the CLIP architecture and how is it used in computer vision?",
    "Explain the variational lower bound and its connection to Jensen's inequality"
]

for i, query in enumerate(evaluation_queries, 1):
    print(f"\n{'='*80}")
    print(f"Evaluation Query {i}")
    print(f"{'='*80}\n")
    print(f"Question: {query}\n")
    
    # Use nano-graphrag with local mode
    result = engine.answer_query(query, use_nano_graphrag=True, mode="local")
    
    print("Answer:")
    print(result["answer"])
    print(f"\nConcepts used: {result['concepts_used']}")
    print(f"Mode: {result['mode']}")
    
    if result['citations']:
        print("\nCitations:")
        for citation in result['citations']:
            print(f"  - {citation['title']} ({citation['type']})")

INFO:socratic.query.hybrid_engine:Processing hybrid query: How do attention mechanisms work in transformers? Can you show me a Python implementation?
INFO:socratic.graph.nano_graphrag_wrapper:Querying nano-graphrag: How do attention mechanisms work in transformers? Can you show me a Python implementation?
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/embeddings "HTTP/1.1 200 OK"
INFO:nano-graphrag:Using 20 entites, 1 communities, 9 relations, 4 text units



Evaluation Query 1

Question: How do attention mechanisms work in transformers? Can you show me a Python implementation?



INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:socratic.query.engine:Identified 2 relevant concepts for query
INFO:socratic.query.engine:Retrieved context: 2 concepts, 1 prerequisites, 2 resources, 0 examples
INFO:socratic.query.hybrid_engine:Processing hybrid query: What is the CLIP architecture and how is it used in computer vision?
INFO:socratic.graph.nano_graphrag_wrapper:Querying nano-graphrag: What is the CLIP architecture and how is it used in computer vision?
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/embeddings "HTTP/1.1 200 OK"
INFO:nano-graphrag:Using 20 entites, 2 communities, 9 relations, 5 text units


Answer:
Sure! Attention mechanisms are pivotal components of transformers, enabling the model to weigh the importance of different parts of its input when making predictions. Let's break down how an attention mechanism works and then look at a simple implementation using PyTorch.

### How Attention Mechanisms Work in Transformers

1. **Query (Q), Key (K), Value (V) Extraction**: The input sequence is transformed into three matrices, Q, K, V.
2. **Attention Scores Calculation**: The model computes attention scores by taking the dot product of the Query vectors with the respective Key vectors.
3. **Softmax Activation**: These scores are passed through a softmax function to turn them into weights.
4. **Weighted Sum**: For each position in the sequence, you get a weighted sum (via the softmax) over all sequences to compute the final output.

### Implementation

Let's see how this can be implemented using PyTorch:

```python
import torch
from torch import nn

class Attention(nn.Module):
   

INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:socratic.query.engine:Identified 3 relevant concepts for query
INFO:socratic.query.engine:Retrieved context: 3 concepts, 0 prerequisites, 3 resources, 0 examples
INFO:socratic.query.hybrid_engine:Processing hybrid query: Explain the variational lower bound and its connection to Jensen's inequality
INFO:socratic.graph.nano_graphrag_wrapper:Querying nano-graphrag: Explain the variational lower bound and its connection to Jensen's inequality
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/embeddings "HTTP/1.1 200 OK"
INFO:nano-graphrag:Using 20 entites, 1 communities, 5 relations, 4 text units


Answer:
CLIP (Contrastive Language-Image Pre-training) is a pretrained model for understanding both text and images. It was developed by researchers at Stanford University and OpenAI, with the original model released publicly as CLIP by OpenAI.

### Key Characteristics of CLIP:
1. **Pretraining Task**: CLIP is trained on a large dataset containing millions of text-image pairs to learn the relationship between words and their corresponding visual representations.
2. **Self-Supervised Learning**: The training process involves predicting whether given pairs of texts and images are describing the same content (positive) or different contents (negative).
3. **Multimodal Understanding**: As it's trained on both textual and visual data, CLIP can effectively understand concepts across these modalities.

### Architecture:
- **Text Encoder**: Converts text into a dense vector representation.
- **Image Encoder**: Converts images into dense vector representations, often through vision transformers

INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:socratic.query.engine:Identified 3 relevant concepts for query
INFO:socratic.query.engine:Retrieved context: 3 concepts, 0 prerequisites, 3 resources, 0 examples


Answer:
The **variational lower bound** is a concept frequently used in machine learning, particularly within probabilistic inference methods such as Variational Bayes (VB) and its variants. This method provides a way to approximate complex posterior distributions by introducing an auxiliary distribution, known as the variational distribution or approximating distribution.

### Definition of Variational Lower Bound

Given a model with parameters \(\theta\) and data \(x\), the goal is often to infer the posterior distribution over some latent variables \(z\). The true posterior is given by:

\[ p(z|x; \theta) = \frac{p(x, z; \theta)}{p(x; \theta)}, \]

where:
- \( p(x, z; \theta) \) is the joint probability of the data and the latents.
- \( p(x; \theta) \) is the marginal likelihood or evidence.

However, calculating this exact posterior can be intractable due to the presence of the normalizing constant (the marginal likelihood in the denominator). Therefore, we construct a simpler appr

## Save Results

In [14]:
# Save hybrid graph
hybrid_builder.save()
print("✅ Hybrid graph saved")

# Graph is now ready for use in main chat interface!

INFO:socratic.graph.nano_graphrag_wrapper:Saved custom metadata to ./data/hybrid_graph/erica_metadata.json
INFO:socratic.graph.hybrid_builder:Saved hybrid knowledge graph


✅ Hybrid graph saved


## Summary

### What We Built

1. **nano-graphrag wrapper**: Direct integration with nano-graphrag for document retrieval
2. **Hybrid builder**: Combines nano-graphrag + custom concept graph
3. **Hybrid query engine**: Uses both retrieval strategies

### When to Use What

- **nano-graphrag (local)**: Focused retrieval, good for specific questions
- **nano-graphrag (global)**: Broad retrieval, good for high-level questions  
- **Custom scaffolded**: Educational scaffolding with prerequisites
- **Hybrid**: Best of both worlds!

### Next Steps

1. Use hybrid graph in chat interface
2. Evaluate different query modes
3. Document system prompts (from nano-graphrag)
4. Create visualizations of both graphs